In [ ]:
from pathlib import Path
import pandas as pd

ruta = Path("../../F2/data/processed/licitaciones_filtradas_provisional.csv.gz")
if not ruta.is_file():
    ruta = Path("../../F2/notebooks/resultados_f2/licitaciones_filtradas_provisional.csv.gz")

df = pd.read_csv(ruta)

print(df.shape)
print(df.columns.tolist())

(41607, 18)
['CodigoExterno', 'Estado', 'NombreOrganismo', 'sector', 'RegionUnidad', 'Tipo', 'Moneda Adquisicion', 'FechaCierre', 'FechaPublicacion', 'MontoEstimado', 'EsDesierta', 'MontoEstimadoOriginal', 'MontoEstimadoNumerico', 'FechaPublicacionOriginal', 'FechaCierreOriginal', 'DiasPublicacionCierre', 'MontoEstimadoImputado', 'MontoFueImputado']


In [2]:
# Cuántos montos faltan en total
total_faltantes = df["MontoEstimadoNumerico"].isna().sum()
print("Faltantes totales:", total_faltantes)

# Cuántos faltan en cada grupo
tabla = df.groupby("EsDesierta")["MontoEstimadoNumerico"].agg(
    total_licitaciones="size",
    faltantes=lambda s: s.isna().sum(),
)

# Qué porcentaje de cada grupo representa
tabla["porcentaje"] = (tabla["faltantes"] / tabla["total_licitaciones"] * 100).round(3)

print(tabla)

Faltantes totales: 7
            total_licitaciones  faltantes  porcentaje
EsDesierta                                           
0                        35622          6       0.017
1                         5985          1       0.017


## Decisión sobre la imputación del monto

Problema: el monto es una de las variables que queremos comparar entre licitaciones desiertas y adjudicadas. Hay 7 licitaciones sin monto y se llenan con la mediana. El riesgo es que, si esos faltantes estuvieran casi todos en un solo grupo, la mediana haría parecer iguales a los dos grupos y se perdería la diferencia que buscamos.

Lo que revisé: conté cuántos montos faltan en cada grupo. Comparé porcentajes y no cantidades, porque hay muchas más adjudicadas (35.622) que desiertas (5.985).

Resultado: faltan 7 montos de 41.607 licitaciones. En las adjudicadas faltan 6 (0,017 %) y en las desiertas falta 1 (0,017 %). El porcentaje es igual en los dos grupos, así que los faltantes no se concentran en uno solo.

Otra opción: excluir del análisis de monto las licitaciones sin dato. Esto sería lo recomendable si los faltantes se concentraran en un grupo.

Decisión: mantener la imputación con la mediana, porque los faltantes están repartidos de forma pareja y son muy pocos.

Limitación: con solo 7 casos, el dato de las desiertas depende de una sola licitación, así que hay que interpretarlo con cuidado.

Nota: hice esta revisión siguiendo la guía del profesor, con apoyo de IA para entender los pasos y escribir el código. No soy informático y estoy aprendiendo programación recientemente.